In [1]:
import pandas as pd
import os
import time
import requests
import spotipy
from dotenv import load_dotenv
from spotipy.oauth2 import SpotifyClientCredentials

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [2]:
load_dotenv()
client_id = os.getenv('SPOTIPY_CLIENT_ID')
client_secret = os.getenv('SPOTIPY_CLIENT_SECRET')
sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(
    client_id=client_id, 
    client_secret=client_secret
))

In [3]:
def music_data_reader(file):
    music_data = pd.read_csv(file)
    return music_data

In [4]:
music_data = music_data_reader('C:/Users/maxph/OneDrive/Desktop/CLARIFY/SongList.csv')
music_data

,Song,Artist,Release_Year
0,Vogue,Madonna,1990
1,Enjoy the Silence,Depeche Mode,1990
2,Ice Ice Baby,Vanilla Ice,1990
3,Black or White,Michael Jackson,1991
4,Smells Like Teen Spirit,Nirvana,1991
5,Losing My Religion,R.E.M.,1991
6,Baby Got Back,Sir Mix-a-Lot,1992
7,I Will Always Love You,Whitney Houston,1992
8,Creep,Radiohead,1992
9,C.R.E.A.M.,Wu-Tang Clan,1993


In [5]:
def fetch_spotify_id(row):
    # Convert to string in case of missing data (NaN)
    song = str(row['Song']) 
    artist = str(row['Artist'])
    
    # Format the query specifically for the Spotify search engine
    query = f"track:{song} artist:{artist}"
    
    try:
        # Search for 1 track matching the query
        results = sp.search(q=query, type='track', limit=1)
        items = results['tracks']['items']
        
        # If the list is not empty, extract the ID
        if items:
            return items[0]['id']
        else:
            print(f"No match found for: {song} by {artist}")
            return None
            
    except Exception as e:
        print(f"API Error fetching {song}: {e}")
        return None

In [6]:
music_data['SONG_ID'] = music_data.apply(fetch_spotify_id, axis=1)
music_data

,Song,Artist,Release_Year,SONG_ID
0,Vogue,Madonna,1990,7j5TIXPi0cCbSSqItmbyZy
1,Enjoy the Silence,Depeche Mode,1990,0yp3TvJNlG50Q4tAHWNCRm
2,Ice Ice Baby,Vanilla Ice,1990,3XVozq1aeqsJwpXrEZrDJ9
3,Black or White,Michael Jackson,1991,7EsjkelQuoUlJXEw7SeVV4
4,Smells Like Teen Spirit,Nirvana,1991,5ghIJDpPoe3CfHMGu71E6T
5,Losing My Religion,R.E.M.,1991,31AOj9sFz2gM0O3hMARRBx
6,Baby Got Back,Sir Mix-a-Lot,1992,1SAkL1mYNJlaqnBQxVZrRl
7,I Will Always Love You,Whitney Houston,1992,4eHbdreAnSOrDDsFfc4Fpm
8,Creep,Radiohead,1992,70LcF31zb1H0PyJoS1Sx1r
9,C.R.E.A.M.,Wu-Tang Clan,1993,119c93MHjrDLJTApCVGpvx


In [7]:
def fetch_lyrics(row):
    # Use your original Song and Artist columns for the search
    song = str(row['Song'])
    artist = str(row['Artist'])
    
    # Skip if missing data
    if pd.isna(song) or pd.isna(artist):
        return pd.Series({'Plain_Lyrics': None, 'Synced_Lyrics': None})

    # The LRCLIB Search Endpoint
    url = "https://lrclib.net/api/search"
    
    # requests will automatically format these into the URL
    params = {
        "track_name": song,
        "artist_name": artist
    }
    
    # The API docs politely ask for a User-Agent, so we will provide a generic one
    headers = {
        "User-Agent": "DataAnalysisScript/1.0"
    }

    try:
        response = requests.get(url, params=params, headers=headers)
        
        if response.status_code == 200:
            data = response.json()
            
            # The search API returns a list of matches. 
            # If the list is not empty, we take the top match (index 0).
            if len(data) > 0:
                top_match = data[0]
                return pd.Series({
                    'Plain_Lyrics': top_match.get('plainLyrics'),
                    'Synced_Lyrics': top_match.get('syncedLyrics')
                })
            else:
                print(f"No lyrics found for: {song} by {artist}")
        else:
            print(f"API Error {response.status_code} for {song}: {response.text}")
            
    except Exception as e:
        print(f"Network error fetching lyrics for {song}: {e}")
        
    # Politeness delay (even without rate limits, it's good practice)
    time.sleep(0.5)
    
    # Return empty if it fails
    return pd.Series({'Plain_Lyrics': None, 'Synced_Lyrics': None})

In [9]:
print("Searching LRCLIB for lyrics...")
lyrics_features = music_data.apply(fetch_lyrics, axis=1)
music_data = pd.concat([music_data, lyrics_features], axis=1)
music_data

Searching LRCLIB for lyrics...
Network error fetching lyrics for Watermelon Sugar: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))


KeyboardInterrupt: 

In [ ]:
music_data.to_csv('C:/Users/maxph/OneDrive/Desktop/CLARIFY/DS3-CLARIFY/TrainingData/lyrics_features.csv', index=False)